In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
import pandas as pd

In [3]:
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        #структура нейросети: input_size - 128 - 64 - 32 - 1
        self.layers = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.layers(x)

In [4]:
traffic = pd.read_csv('../../data/df_train_traffic_encoded_scaled.csv')

In [5]:
#делим на 60000, потому что остальные переменные в районе -1 и 1, веса становятся слишком большими и результаты становятся нестабильными
traffic['Трафик'] = traffic['Трафик'] / 60000
traffic

,Трафик,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час",month_sin,month_cos,pca_1,pca_2,pca_3,pca_4,Населенный пункт_fold_traffic,Регион_fold_traffic,"Дата открытия, категориальный_Новый","Дата открытия, категориальный_Открыт давно","Дата открытия, категориальный_Средний по возрасту","Торговая площадь, категориальный_Большой","Торговая площадь, категориальный_Маленький","Торговая площадь, категориальный_Очень большой","Торговая площадь, категориальный_Средний"
0,0.994367,-0.183291,-0.634931,-0.658506,0.29525,-8.660254e-01,5.000000e-01,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
1,0.944567,-0.183291,-0.634931,-0.658506,0.29525,5.000000e-01,-8.660254e-01,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
2,0.858133,-0.183291,-0.634931,-0.658506,0.29525,5.000000e-01,8.660254e-01,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
3,0.944883,-0.183291,-0.634931,-0.658506,0.29525,1.224647e-16,-1.000000e+00,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
4,0.968800,-0.183291,-0.634931,-0.658506,0.29525,-5.000000e-01,-8.660254e-01,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232664,0.861267,-0.187406,-0.712807,-0.009117,0.52820,-8.660254e-01,5.000000e-01,-1.678946,0.112034,-0.095371,-0.163764,-1.463815,-0.786469,1,0,0,0,0,0,1
232665,0.858600,-0.187406,-0.712807,-0.009117,0.52820,-5.000000e-01,8.660254e-01,-1.678946,0.112034,-0.095371,-0.163764,-1.463815,-0.786469,1,0,0,0,0,0,1
232666,0.826550,-0.187406,-0.712807,-0.009117,0.52820,-1.000000e+00,-1.836970e-16,-1.678946,0.112034,-0.095371,-0.163764,-1.463815,-0.786469,1,0,0,0,0,0,1
232667,0.868583,-0.187406,-0.712807,-0.009117,0.52820,-2.449294e-16,1.000000e+00,-1.678946,0.112034,-0.095371,-0.163764,-1.463815,-0.786469,1,0,0,0,0,0,1


In [6]:
mlp = MLP(input_size=(traffic.shape[1] - 1))
#функци потерь, сочетающая RMSE и MAE
criterion = nn.HuberLoss(delta=1.0)
optimizer = optim.Adam(mlp.parameters(), lr=0.001)

In [7]:
y_traffic = traffic['Трафик']
x_traffic = traffic.drop(['Трафик'], axis=1)

In [8]:
from sklearn.model_selection import train_test_split

x_train_traffic, x_test_traffic, y_train_traffic, y_test_traffic = train_test_split(x_traffic, y_traffic, test_size=0.2, random_state=598)

In [9]:
#pytorch требует данных в формате тензоров
x_tensor = torch.tensor(x_train_traffic.values, dtype=torch.float32)
y_tensor = torch.tensor(y_train_traffic.values, dtype=torch.float32).reshape(-1, 1)

In [10]:
from torch.utils.data import DataLoader, TensorDataset
dataset = TensorDataset(x_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=100, shuffle=True)

In [11]:
from torch.nn.functional import l1_loss

In [12]:
x_test_tensor = torch.tensor(x_test_traffic.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_traffic.values, dtype=torch.float32).reshape(-1, 1)

In [13]:
for epoch in range(300):
    mlp.train()
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        
        predictions = mlp(batch_x)
        loss = criterion(predictions, batch_y)
        
        loss.backward()
        optimizer.step()

    mlp.eval()    
    with torch.no_grad():
        train_preds = mlp(x_tensor)
        rmse_train = torch.sqrt(torch.mean((train_preds - y_tensor) ** 2)).item() * 60000
        mae_train = l1_loss(train_preds * 60000, y_tensor * 60000).item()
        
        test_preds = mlp(x_test_tensor)
        rmse_test = torch.sqrt(torch.mean((test_preds - y_test_tensor) ** 2)).item() * 60000
        mae_test = l1_loss(test_preds * 60000, y_test_tensor * 60000).item()
        
    print(f"Эпоха {epoch}"
          f" Train RMSE: {rmse_train:7.2f}, MAE: {mae_train:7.2f}"
          f" Test RMSE: {rmse_test:7.2f}, MAE: {mae_test:7.2f}")

Эпоха 0 Train RMSE: 9815.39, MAE: 7559.99 Test RMSE: 9885.85, MAE: 7584.33
Эпоха 1 Train RMSE: 9617.46, MAE: 7307.13 Test RMSE: 9687.22, MAE: 7323.46
Эпоха 2 Train RMSE: 9664.03, MAE: 7312.81 Test RMSE: 9740.00, MAE: 7349.65
Эпоха 3 Train RMSE: 9709.60, MAE: 7345.41 Test RMSE: 9793.27, MAE: 7392.78
Эпоха 4 Train RMSE: 9313.78, MAE: 7136.60 Test RMSE: 9451.21, MAE: 7204.24
Эпоха 5 Train RMSE: 9158.98, MAE: 6961.52 Test RMSE: 9317.57, MAE: 7065.61
Эпоха 6 Train RMSE: 9014.67, MAE: 6880.68 Test RMSE: 9201.81, MAE: 6994.28
Эпоха 7 Train RMSE: 8838.81, MAE: 6755.08 Test RMSE: 9054.50, MAE: 6893.18
Эпоха 8 Train RMSE: 8642.23, MAE: 6620.04 Test RMSE: 8880.48, MAE: 6764.82
Эпоха 9 Train RMSE: 8547.81, MAE: 6542.13 Test RMSE: 8805.57, MAE: 6704.91
Эпоха 10 Train RMSE: 8344.44, MAE: 6344.34 Test RMSE: 8618.59, MAE: 6522.98
Эпоха 11 Train RMSE: 8189.36, MAE: 6259.75 Test RMSE: 8495.74, MAE: 6453.39
Эпоха 12 Train RMSE: 8053.62, MAE: 6157.14 Test RMSE: 8362.47, MAE: 6353.39
Эпоха 13 Train RMSE: 7